# SVM 为什么追求最大间隔？

**面试回答主线：**SVM 不满足于把样本分开，而是最大化最近训练样本到决策边界的几何间隔。软间隔以 hinge loss 允许少量违例，并用正则控制边界复杂度。本实验手写线性 SVM 的次梯度更新，观察支持向量和错标点对边界的影响。

## 真实案例

质检系统根据商品外壳划痕分数和传感器偏差判断“合格”或“复检”。复检样本的两个特征通常更大，但边界附近会存在不确定订单。标签用 -1/1 表示，便于直接写出 hinge loss。

In [1]:
import numpy as np  # 导入 NumPy 以手写 SVM 次梯度优化。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的数值显示。
item = np.array(['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'V01', 'V02'])  # 构造质检商品编号。
feature = np.array([[0.2, 0.1], [0.3, 0.4], [0.5, 0.2], [0.6, 0.5], [1.5, 1.4], [1.7, 1.5], [1.9, 1.6], [2.1, 1.9], [0.4, 0.6], [1.8, 1.7]], dtype=float)  # 记录划痕分数和传感器偏差。
label = np.array([-1, -1, -1, -1, 1, 1, 1, 1, -1, 1], dtype=float)  # 记录合格负类和复检正类标签。
train_index = np.arange(8)  # 将八个历史质检样本用于训练。
valid_index = np.arange(8, 10)  # 将两个新批次商品用于回放验证。
print('商品 | 划痕分 | 偏差分 | 质检标签')  # 输出质检数据表头。
for index in range(len(item)):  # 逐条展示业务样本。
    text_label = '复检' if label[index] == 1 else '合格'  # 将数值标签翻译为业务动作。
    print(f'{item[index]} | {feature[index, 0]:6.1f} | {feature[index, 1]:6.1f} | {text_label}')  # 输出一条质检记录。

商品 | 划痕分 | 偏差分 | 质检标签
Q01 |    0.2 |    0.1 | 合格
Q02 |    0.3 |    0.4 | 合格
Q03 |    0.5 |    0.2 | 合格
Q04 |    0.6 |    0.5 | 合格
Q05 |    1.5 |    1.4 | 复检
Q06 |    1.7 |    1.5 | 复检
Q07 |    1.9 |    1.6 | 复检
Q08 |    2.1 |    1.9 | 复检
V01 |    0.4 |    0.6 | 合格
V02 |    1.8 |    1.7 | 复检


## Baseline / 基线

基线仅按划痕分大于 1 判断复检，忽略传感器偏差，也没有利用间隔信息。

In [2]:
baseline_pred = np.where(feature[valid_index, 0] > 1.0, 1.0, -1.0)  # 用单特征阈值生成规则基线预测。
baseline_accuracy = float(np.mean(baseline_pred == label[valid_index]))  # 计算规则基线回放准确率。
print('规则基线预测:', baseline_pred.astype(int))  # 输出规则基线的正负类结果。
print(f'规则基线准确率={baseline_accuracy:.3f}')  # 输出基线指标。

规则基线预测: [-1  1]
规则基线准确率=1.000


In [3]:
mean = feature[train_index].mean(axis=0)  # 仅从训练样本计算特征均值。
std = feature[train_index].std(axis=0)  # 仅从训练样本计算特征尺度。
x_train = (feature[train_index] - mean) / std  # 标准化训练特征使间隔具有可比尺度。
x_valid = (feature[valid_index] - mean) / std  # 用训练统计量标准化新批次特征。
def fit_linear_svm(x, y, regularization, steps):  # 定义线性软间隔 SVM 的次梯度训练函数。
    weight = np.zeros(x.shape[1])  # 初始化超平面法向量。
    bias = 0.0  # 初始化决策边界偏置。
    for step in range(steps):  # 迭代最小化正则项加 hinge loss。
        margin = y * (x @ weight + bias)  # 计算每个样本的函数间隔。
        active = margin < 1.0  # 找到仍在间隔内或被错分的样本。
        grad_weight = regularization * weight - (x[active] * y[active, None]).sum(axis=0) / len(x)  # 计算权重的次梯度。
        grad_bias = -y[active].sum() / len(x)  # 计算偏置的次梯度。
        weight -= 0.12 * grad_weight  # 更新超平面法向量。
        bias -= 0.12 * grad_bias  # 更新边界偏置。
    return weight, bias  # 返回训练后的边界参数。
print('标准化后训练特征前两行:', np.round(x_train[:2], 2))  # 展示 SVM 实际使用的特征中间量。

标准化后训练特征前两行: [[-1.24 -1.26]
 [-1.1  -0.82]]


In [4]:
weight, bias = fit_linear_svm(x_train, label[train_index], 0.18, 500)  # 训练带适中正则的线性 SVM。
train_margin = label[train_index] * (x_train @ weight + bias)  # 计算训练样本的函数间隔。
valid_score = x_valid @ weight + bias  # 计算新批次商品的决策得分。
svm_pred = np.where(valid_score >= 0.0, 1.0, -1.0)  # 以零边界将得分转为质检动作。
svm_accuracy = float(np.mean(svm_pred == label[valid_index]))  # 计算 SVM 的回放准确率。
support_mask = train_margin <= 1.05  # 近似标记决定间隔的支持向量候选。
print('SVM 权重:', np.round(weight, 3), '偏置:', round(bias, 3))  # 输出学习到的超平面参数。
print('训练函数间隔:', np.round(train_margin, 3))  # 输出解释 hinge loss 的中间量。
print('支持向量候选:', item[train_index][support_mask].tolist())  # 输出靠近边界的关键商品。
print(f'SVM 回放准确率={svm_accuracy:.3f}')  # 输出主要模型指标。

SVM 权重: [0.737 0.795] 偏置: 0.045
训练函数间隔: [1.873 1.417 1.45  0.994 0.983 1.304 1.625 2.182]
支持向量候选: ['Q04', 'Q05']
SVM 回放准确率=1.000


## 结果解读

间隔小于 1 的样本贡献 hinge loss，远离边界且已正确分类的样本不再推动同方向更新。几何间隔还需除以权重范数，因此不能只比较原始 score 的大小。支持向量是当前边界的关键证据，不代表它们一定是脏样本。

In [5]:
geometric_margin = train_margin / np.linalg.norm(weight)  # 将函数间隔除以权重范数得到几何间隔比例。
print('商品 | 函数间隔 | 几何间隔 | 是否激活hinge')  # 输出间隔诊断表头。
for local_index, global_index in enumerate(train_index):  # 逐条展示历史样本的间隔。
    print(f'{item[global_index]} | {train_margin[local_index]:8.3f} | {geometric_margin[local_index]:8.3f} | {bool(train_margin[local_index] < 1.0)}')  # 输出函数间隔、几何间隔和 hinge 状态。
print('结果：最大间隔并不是追求训练分数最高，而是让边界远离最危险的已知样本。')  # 总结最大间隔的业务含义。

商品 | 函数间隔 | 几何间隔 | 是否激活hinge
Q01 |    1.873 |    1.728 | False
Q02 |    1.417 |    1.307 | False
Q03 |    1.450 |    1.338 | False
Q04 |    0.994 |    0.917 | True
Q05 |    0.983 |    0.907 | True
Q06 |    1.304 |    1.203 | False
Q07 |    1.625 |    1.499 | False
Q08 |    2.182 |    2.013 | False
结果：最大间隔并不是追求训练分数最高，而是让边界远离最危险的已知样本。


## 失败案例与修复

下面故意把一个合格商品错标为复检，并把正则强度降得很低。模型会过度迁就这条异常记录，支持向量和边界都被拉动。修复包括复核标注、使用软间隔/类别权重，并在按批次的验证集上选择 C 或正则强度。

In [6]:
noisy_label = label[train_index].copy()  # 复制训练标签以构造人工错标。
noisy_label[2] = 1.0  # 将 Q03 从合格错误标为复检。
bad_weight, bad_bias = fit_linear_svm(x_train, noisy_label, 0.01, 500)  # 用极弱正则拟合错标数据。
bad_margin = noisy_label * (x_train @ bad_weight + bad_bias)  # 计算错标训练下的间隔。
fixed_weight, fixed_bias = fit_linear_svm(x_train, label[train_index], 0.18, 500)  # 恢复核验标签和适中正则重新训练。
print('失败：错标且弱正则的 Q03 间隔=', round(bad_margin[2], 3))  # 展示模型尝试迁就错误标签的中间量。
print('修复：核验标签后的 Q03 真实间隔=', round(train_margin[2], 3))  # 展示恢复后对该样本的正确处理。
print('生产差距：需记录标注版本、按设备/批次切分、校准置信度、支持向量比例和人工复检闭环。')  # 说明真正质检系统的额外控制。

失败：错标且弱正则的 Q03 间隔= -0.493
修复：核验标签后的 Q03 真实间隔= 1.45
生产差距：需记录标注版本、按设备/批次切分、校准置信度、支持向量比例和人工复检闭环。


In [7]:
assert len(item) >= 5  # 保护案例至少包含五个具名质检样本。
assert svm_accuracy >= baseline_accuracy  # 保护双特征 SVM 不弱于单阈值基线。
assert support_mask.sum() > 0  # 保护实验存在可解释的边界关键样本。
assert not np.allclose(weight, bad_weight)  # 保护错标会改变边界这一失败现象。